# Titanic - Machine Learning from Disaster
## model_experiment.ipynb

In [1]:
1

1

In [2]:
!pip install mlflow category_encoders dagshub scikit-learn xgboost lightgbm --quiet


[notice] A new release of pip available: 22.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [ ]:
import os
import warnings
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

from sklearn.feature_selection import RFE
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.metrics import (
    accuracy_score, roc_auc_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, RocCurveDisplay
)

import category_encoders as ce
import mlflow
import mlflow.sklearn

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 50)

## MLflow / DagsHub Setup

Set your DagsHub credentials below **or** leave `USE_DAGSHUB = False` to log locally.

In [ ]:
USE_DAGSHUB = False

DAGSHUB_USERNAME = "YOUR_DAGSHUB_USERNAME"
DAGSHUB_REPO     = "YOUR_REPO_NAME"
DAGSHUB_TOKEN    = "e8aa6edaae9cca4d616c2dc4f9f6602767dcede1"

if USE_DAGSHUB:
    os.environ["MLFLOW_TRACKING_USERNAME"] = DAGSHUB_USERNAME
    os.environ["MLFLOW_TRACKING_PASSWORD"] = DAGSHUB_TOKEN
    tracking_uri = f"https://dagshub.com/{DAGSHUB_USERNAME}/{DAGSHUB_REPO}.mlflow"
    mlflow.set_tracking_uri(tracking_uri)
else:
    mlflow.set_tracking_uri("mlruns")

EXPERIMENT_NAME = "titanic-experiments"
mlflow.set_experiment(EXPERIMENT_NAME)
print("Tracking URI:", mlflow.get_tracking_uri())

## Data Loading

In [ ]:
train_raw = pd.read_csv("data/train.csv")
test_raw  = pd.read_csv("data/test.csv")

print("Train shape:", train_raw.shape)
print("Test shape: ", test_raw.shape)
train_raw.head()

In [ ]:
print(train_raw.isnull().sum())

## Train / Validation Split

Split before any encoding so encoders (WOE, OHE categories) are fit **only on training data**,  
preventing leakage from the validation set.

In [ ]:
VAL_SIZE   = 0.2
RANDOM_SEED = 42
TARGET      = 'Survived'

train_split, val_split = train_test_split(
    train_raw,
    test_size=VAL_SIZE,
    random_state=RANDOM_SEED,
    stratify=train_raw[TARGET]
)

train_split = train_split.reset_index(drop=True)
val_split   = val_split.reset_index(drop=True)

print(f"Train split: {train_split.shape}  |  Val split: {val_split.shape}")
print(f"Survival rate — train: {train_split[TARGET].mean():.3f}  val: {val_split[TARGET].mean():.3f}")

---
# Data Cleaning

In [ ]:
def extract_title(name: str) -> str:
    title = name.split(',')[1].split('.')[0].strip()
    rare = {'Lady','Countess','Capt','Col','Don','Dr','Major','Rev','Sir','Jonkheer','Dona'}
    if title in rare:
        return 'Rare'
    if title == 'Mlle':
        return 'Miss'
    if title in ('Ms', 'Mme'):
        return 'Mrs'
    return title


def clean(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    df['Title'] = df['Name'].apply(extract_title)

    age_medians = df.groupby(['Pclass', 'Sex'])['Age'].transform('median')
    df['Age'] = df['Age'].fillna(age_medians)
    df['Age'] = df['Age'].fillna(df['Age'].median())

    df['Fare'] = df['Fare'].fillna(df.groupby('Pclass')['Fare'].transform('median'))

    df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])

    df['Deck'] = df['Cabin'].str[0].fillna('Unknown')

    df.drop(columns=['Cabin', 'Name', 'Ticket', 'PassengerId'], errors='ignore', inplace=True)

    return df


train_clean = clean(train_split)
val_clean   = clean(val_split)
test_clean  = clean(test_raw)

print("Missing after cleaning (train):")
print(train_clean.isnull().sum())
train_clean.head()

---
# Feature Engineering
## New Features

In [ ]:
def add_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
    df['IsAlone']    = (df['FamilySize'] == 1).astype(int)
    df['AgeBin']     = pd.cut(df['Age'],  bins=[0,12,18,35,60,120],
                              labels=['Child','Teen','Adult','MidAge','Senior'])
    df['FareBin']    = pd.qcut(df['Fare'], q=4,
                               labels=['Low','Mid','High','VHigh'], duplicates='drop')
    return df


train_fe = add_features(train_clean)
val_fe   = add_features(val_clean)
test_fe  = add_features(test_clean)

train_fe.head()

## One-Hot Encoding (OHE)

Categories learned from **train only**. Val and test are aligned to those columns (missing → 0).

In [ ]:
OHE_COLS     = ['Sex', 'Embarked', 'Title', 'Deck', 'AgeBin', 'FareBin']
numeric_cols = ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare', 'FamilySize', 'IsAlone']


def apply_ohe(train_df: pd.DataFrame, other_dfs: list, cols: list):
    train_dummies = pd.get_dummies(train_df[cols], columns=cols, drop_first=False)
    ohe_columns   = train_dummies.columns.tolist()
    results = [train_dummies]
    for df in other_dfs:
        dummies  = pd.get_dummies(df[cols], columns=cols, drop_first=False)
        aligned  = dummies.reindex(columns=ohe_columns, fill_value=0)
        results.append(aligned.reset_index(drop=True))
    return results


ohe_parts = apply_ohe(train_fe, [val_fe, test_fe], OHE_COLS)
ohe_train_enc, ohe_val_enc, ohe_test_enc = ohe_parts

y_train = train_fe[TARGET].reset_index(drop=True)
y_val   = val_fe[TARGET].reset_index(drop=True)

X_ohe_train = pd.concat([train_fe[numeric_cols].reset_index(drop=True), ohe_train_enc], axis=1)
X_ohe_val   = pd.concat([val_fe[numeric_cols].reset_index(drop=True),   ohe_val_enc],   axis=1)
X_ohe_test  = pd.concat([test_fe[numeric_cols].reset_index(drop=True),  ohe_test_enc],  axis=1)

print("OHE train shape:", X_ohe_train.shape)
print("OHE val shape:  ", X_ohe_val.shape)
X_ohe_train.head()

## Weight of Evidence (WOE) Encoding

Encoder fit on **train only**, then applied to val and test.  
WOE = ln(P(X|Y=1) / P(X|Y=0)) — natural for binary classification.

In [ ]:
WOE_COLS = ['Sex', 'Embarked', 'Title', 'Deck', 'AgeBin', 'FareBin', 'Pclass']

woe_encoder = ce.WOEEncoder(cols=WOE_COLS, regularization=1.0)


def prep_woe(df: pd.DataFrame) -> pd.DataFrame:
    out = df[numeric_cols + WOE_COLS].copy()
    for c in ['AgeBin', 'FareBin']:
        out[c] = out[c].astype(str)
    return out


woe_encoder.fit(prep_woe(train_fe), y_train)

X_woe_train = woe_encoder.transform(prep_woe(train_fe))
X_woe_val   = woe_encoder.transform(prep_woe(val_fe))
X_woe_test  = woe_encoder.transform(prep_woe(test_fe))

print("WOE train shape:", X_woe_train.shape)
print("WOE val shape:  ", X_woe_val.shape)
X_woe_train.head()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
X_ohe_train.hist(ax=axes[0], bins=20, layout=(1,1))
axes[0].set_title('OHE Feature Distributions (train)')
X_woe_train.hist(ax=axes[1], bins=20, layout=(1,1))
axes[1].set_title('WOE Feature Distributions (train)')
plt.tight_layout()
plt.savefig('feature_distributions.png', dpi=80)
plt.show()

---
# Feature Selection
## Recursive Feature Elimination (RFE)

RFE fit on **train split only**.

In [ ]:
def run_rfe(X_tr: pd.DataFrame, y_tr: pd.Series, n_features: int, label: str):
    estimator = RandomForestClassifier(n_estimators=100, random_state=42)
    scaler    = StandardScaler()
    X_scaled  = scaler.fit_transform(X_tr)
    rfe       = RFE(estimator=estimator, n_features_to_select=n_features, step=1)
    rfe.fit(X_scaled, y_tr)
    selected  = X_tr.columns[rfe.support_].tolist()
    ranking   = pd.Series(rfe.ranking_, index=X_tr.columns).sort_values()

    print(f"\n[{label}] Selected {n_features} features:")
    print(selected)

    plt.figure(figsize=(10, 5))
    ranking.head(n_features + 5).plot(kind='bar')
    plt.title(f'RFE Feature Ranking — {label}')
    plt.ylabel('Rank (lower = better)')
    plt.tight_layout()
    plt.savefig(f'rfe_ranking_{label}.png', dpi=80)
    plt.show()

    return selected


N_FEATURES_OHE = 15
N_FEATURES_WOE = 10

selected_ohe = run_rfe(X_ohe_train, y_train, N_FEATURES_OHE, 'OHE')
selected_woe = run_rfe(X_woe_train, y_train, N_FEATURES_WOE, 'WOE')

In [ ]:
X_ohe_tr_sel   = X_ohe_train[selected_ohe]
X_ohe_val_sel  = X_ohe_val[selected_ohe]
X_ohe_test_sel = X_ohe_test[selected_ohe]

X_woe_tr_sel   = X_woe_train[selected_woe]
X_woe_val_sel  = X_woe_val[selected_woe]
X_woe_test_sel = X_woe_test[selected_woe]

print("OHE — train:", X_ohe_tr_sel.shape, " val:", X_ohe_val_sel.shape)
print("WOE — train:", X_woe_tr_sel.shape, " val:", X_woe_val_sel.shape)

---
# Training

Each run:
- CV metrics computed on **train split** (5-fold)
- Val metrics computed on the **held-out validation split**
- Logs params, metrics, confusion matrix, ROC curve, feature importance

In [ ]:
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


def log_run(
    run_name: str,
    model,
    X_tr: pd.DataFrame,
    y_tr: pd.Series,
    X_vl: pd.DataFrame,
    y_vl: pd.Series,
    encoding: str,
    extra_params: dict = None,
    register: bool = False,
):
    X_tr_arr = X_tr.values.astype(float)
    X_vl_arr = X_vl.values.astype(float)

    acc_scores = cross_val_score(model, X_tr_arr, y_tr, cv=CV, scoring='accuracy')
    auc_scores = cross_val_score(model, X_tr_arr, y_tr, cv=CV, scoring='roc_auc')
    f1_scores  = cross_val_score(model, X_tr_arr, y_tr, cv=CV, scoring='f1')

    model.fit(X_tr_arr, y_tr)
    y_pred = model.predict(X_vl_arr)
    y_prob = model.predict_proba(X_vl_arr)[:, 1] if hasattr(model, 'predict_proba') else None

    with mlflow.start_run(run_name=run_name):
        params = {
            'model':      type(model).__name__,
            'encoding':   encoding,
            'n_features': X_tr.shape[1],
            'val_size':   len(y_vl),
        }
        if extra_params:
            params.update(extra_params)
        mlflow.log_params(params)

        mlflow.log_metrics({
            'cv_acc_mean':  acc_scores.mean(),
            'cv_acc_std':   acc_scores.std(),
            'cv_auc_mean':  auc_scores.mean(),
            'cv_auc_std':   auc_scores.std(),
            'cv_f1_mean':   f1_scores.mean(),
            'cv_f1_std':    f1_scores.std(),
            'val_accuracy': accuracy_score(y_vl, y_pred),
            'val_f1':       f1_score(y_vl, y_pred),
            'val_auc':      roc_auc_score(y_vl, y_prob) if y_prob is not None else 0.0,
        })

        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        ConfusionMatrixDisplay(confusion_matrix(y_vl, y_pred)).plot(ax=axes[0], colorbar=False)
        axes[0].set_title('Confusion Matrix (val)')
        if y_prob is not None:
            RocCurveDisplay.from_predictions(y_vl, y_prob, ax=axes[1])
            axes[1].set_title('ROC Curve (val)')
        plt.suptitle(run_name)
        plt.tight_layout()
        plot_path = f'{run_name.replace(" ", "_")}_plots.png'
        plt.savefig(plot_path, dpi=80)
        plt.show()
        mlflow.log_artifact(plot_path)

        if hasattr(model, 'feature_importances_'):
            imp = pd.Series(model.feature_importances_, index=X_tr.columns).sort_values(ascending=False)
            fig2, ax2 = plt.subplots(figsize=(10, 4))
            imp.head(15).plot(kind='bar', ax=ax2)
            ax2.set_title(f'Feature Importance — {run_name}')
            plt.tight_layout()
            imp_path = f'{run_name.replace(" ", "_")}_importance.png'
            plt.savefig(imp_path, dpi=80)
            plt.show()
            mlflow.log_artifact(imp_path)

        if register:
            mlflow.sklearn.log_model(
                model, artifact_path='model',
                registered_model_name='titanic-best-model'
            )
        else:
            mlflow.sklearn.log_model(model, artifact_path='model')

        run_id = mlflow.active_run().info.run_id

    print(f"[{run_name}] cv_acc={acc_scores.mean():.4f} cv_auc={auc_scores.mean():.4f} val_auc={roc_auc_score(y_vl, y_prob) if y_prob is not None else 0.0:.4f}  run_id={run_id}")
    return {
        'run_name': run_name,
        'run_id':   run_id,
        'cv_acc':   acc_scores.mean(),
        'cv_auc':   auc_scores.mean(),
        'cv_f1':    f1_scores.mean(),
        'val_auc':  roc_auc_score(y_vl, y_prob) if y_prob is not None else 0.0,
        'val_acc':  accuracy_score(y_vl, y_pred),
    }

### Logistic Regression — OHE vs WOE

In [ ]:
results = []

for C in [0.01, 0.1, 1.0, 10.0]:
    lr = Pipeline([
        ('scaler', StandardScaler()),
        ('clf',    LogisticRegression(C=C, max_iter=1000, random_state=42))
    ])
    results.append(log_run(
        f'LR_OHE_C{C}', lr,
        X_ohe_tr_sel, y_train, X_ohe_val_sel, y_val,
        'OHE', extra_params={'C': C, 'solver': 'lbfgs'}
    ))
    results.append(log_run(
        f'LR_WOE_C{C}', lr,
        X_woe_tr_sel, y_train, X_woe_val_sel, y_val,
        'WOE', extra_params={'C': C, 'solver': 'lbfgs'}
    ))

### Random Forest — OHE vs WOE

In [ ]:
for n_est in [50, 100, 200]:
    for max_depth in [4, 6, None]:
        rf = RandomForestClassifier(
            n_estimators=n_est, max_depth=max_depth,
            min_samples_leaf=2, random_state=42
        )
        results.append(log_run(
            f'RF_OHE_n{n_est}_d{max_depth}', rf,
            X_ohe_tr_sel, y_train, X_ohe_val_sel, y_val,
            'OHE', extra_params={'n_estimators': n_est, 'max_depth': str(max_depth)}
        ))
        results.append(log_run(
            f'RF_WOE_n{n_est}_d{max_depth}', rf,
            X_woe_tr_sel, y_train, X_woe_val_sel, y_val,
            'WOE', extra_params={'n_estimators': n_est, 'max_depth': str(max_depth)}
        ))

### XGBoost — OHE vs WOE

In [ ]:
for lr_val in [0.05, 0.1, 0.2]:
    for max_depth in [3, 5, 7]:
        xgb = XGBClassifier(
            n_estimators=200, learning_rate=lr_val, max_depth=max_depth,
            subsample=0.8, colsample_bytree=0.8,
            use_label_encoder=False, eval_metric='logloss', random_state=42
        )
        results.append(log_run(
            f'XGB_OHE_lr{lr_val}_d{max_depth}', xgb,
            X_ohe_tr_sel, y_train, X_ohe_val_sel, y_val,
            'OHE', extra_params={'learning_rate': lr_val, 'max_depth': max_depth, 'n_estimators': 200}
        ))
        results.append(log_run(
            f'XGB_WOE_lr{lr_val}_d{max_depth}', xgb,
            X_woe_tr_sel, y_train, X_woe_val_sel, y_val,
            'WOE', extra_params={'learning_rate': lr_val, 'max_depth': max_depth, 'n_estimators': 200}
        ))

### LightGBM — OHE vs WOE

In [ ]:
for n_leaves in [15, 31, 63]:
    lgbm = LGBMClassifier(
        n_estimators=200, num_leaves=n_leaves,
        learning_rate=0.05, random_state=42, verbose=-1
    )
    results.append(log_run(
        f'LGBM_OHE_leaves{n_leaves}', lgbm,
        X_ohe_tr_sel, y_train, X_ohe_val_sel, y_val,
        'OHE', extra_params={'num_leaves': n_leaves, 'n_estimators': 200, 'learning_rate': 0.05}
    ))
    results.append(log_run(
        f'LGBM_WOE_leaves{n_leaves}', lgbm,
        X_woe_tr_sel, y_train, X_woe_val_sel, y_val,
        'WOE', extra_params={'num_leaves': n_leaves, 'n_estimators': 200, 'learning_rate': 0.05}
    ))

## Results Summary

In [ ]:
results_df = pd.DataFrame(results).sort_values('val_auc', ascending=False)
print(results_df.to_string(index=False))

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, metric in zip(axes, ['cv_acc', 'cv_auc', 'val_auc']):
    results_df.head(20).plot(x='run_name', y=metric, kind='bar', ax=ax, legend=False)
    ax.set_title(metric)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=90, fontsize=7)
plt.tight_layout()
plt.savefig('results_comparison.png', dpi=80)
plt.show()

## Register Best Model

Best model is selected by `val_auc` on the held-out validation split, then retrained on full train data.

In [ ]:
best = results_df.iloc[0]
print("Best run:", best['run_name'], "| val_auc:", best['val_auc'])

best_encoding = 'OHE' if '_OHE_' in best['run_name'] else 'WOE'
X_tr_best  = X_ohe_tr_sel  if best_encoding == 'OHE' else X_woe_tr_sel
X_val_best = X_ohe_val_sel if best_encoding == 'OHE' else X_woe_val_sel

model_map = {
    'LR':   lambda: Pipeline([('scaler', StandardScaler()),
                               ('clf', LogisticRegression(C=1.0, max_iter=1000, random_state=42))]),
    'RF':   lambda: RandomForestClassifier(n_estimators=100, random_state=42),
    'XGB':  lambda: XGBClassifier(n_estimators=200, use_label_encoder=False, eval_metric='logloss', random_state=42),
    'LGBM': lambda: LGBMClassifier(n_estimators=200, random_state=42, verbose=-1),
}

model_key  = best['run_name'].split('_')[0]
best_model = model_map.get(model_key, model_map['RF'])()

X_full = pd.concat([X_tr_best, X_val_best], axis=0).reset_index(drop=True)
y_full = pd.concat([y_train,   y_val],       axis=0).reset_index(drop=True)
best_model.fit(X_full.values.astype(float), y_full)

log_run(
    f'BEST_{best["run_name"]}',
    best_model,
    X_tr_best, y_train,
    X_val_best, y_val,
    best_encoding,
    register=True
)
print("Model registered as 'titanic-best-model' in MLflow Model Registry.")

In [ ]:
artifacts = {
    'best_run_id':   best['run_id'],
    'best_encoding': best_encoding,
    'selected_ohe':  selected_ohe,
    'selected_woe':  selected_woe,
    'woe_encoder':   woe_encoder,
    'numeric_cols':  numeric_cols,
    'ohe_cols':      OHE_COLS,
    'woe_cols':      WOE_COLS,
}

with open('preprocessing_artifacts.pkl', 'wb') as f:
    pickle.dump(artifacts, f)

print("Artifacts saved. Best run_id:", best['run_id'])